Experimenting with the model

In [54]:
# Use a model here
import torch
from sentence_transformers import SentenceTransformer, models
import multihead_generalized_pooling


In [55]:
import importlib
importlib.reload(multihead_generalized_pooling)
from multihead_generalized_pooling import *


In [56]:
# Step 1: Load the existing SentenceTransformer model
existing_model = SentenceTransformer("sentence-transformers/distiluse-base-multilingual-cased-v2")

# Extract the transformer and the last dense layer
transformer = existing_model[0]  # Transformer (DistilBERT)
dense_layer = existing_model[2]   # Dense layer


# Step 3: Create the new SentenceTransformer model
generalized_pooling = MultiHeadGeneralizedPooling(token_dim=transformer.get_word_embedding_dimension())

# Build the new model using the reused components
model = SentenceTransformer(modules=[transformer, generalized_pooling, dense_layer])

# Step 4: Print the new model architecture
print(model)

# Run the model on an example
sentences = ["First phrase", "Second phrase", "Third phrase darling"]

SentenceTransformer(
  (0): Transformer({'max_seq_length': 128, 'do_lower_case': False}) with Transformer model: DistilBertModel 
  (1): MultiHeadGeneralizedPooling(
    (W1): ModuleList(
      (0): Linear(in_features=768, out_features=3072, bias=True)
    )
    (W2): ModuleList(
      (0): Linear(in_features=3072, out_features=768, bias=True)
    )
    (P): ModuleList(
      (0): Linear(in_features=768, out_features=768, bias=True)
    )
  )
  (2): Dense({'in_features': 768, 'out_features': 512, 'bias': True, 'activation_function': 'torch.nn.modules.activation.Tanh'})
)


In [57]:
print("COMPUTING EMBEDDINGS")
sentence_embeddings = model.encode(sentences)


COMPUTING EMBEDDINGS
projected torch.Size([3, 6, 768])
W1 torch.Size([3, 6, 3072])
W2 torch.Size([3, 6, 768])
attention mask torch.Size([3, 6, 768])


In [58]:
print("EMBEDDINGS COMPUTED WITH INITIALIZED GENERALIZED POOLING")
print(sentence_embeddings)

EMBEDDINGS COMPUTED WITH INITIALIZED GENERALIZED POOLING
[[ 0.05568264  0.07488462 -0.0569736  ... -0.0319594   0.02177089
  -0.01300419]
 [ 0.02329546  0.04503326 -0.04492065 ... -0.02495108  0.04223553
  -0.0437062 ]
 [ 0.02012968 -0.01120749 -0.06453869 ... -0.03425869 -0.03193577
   0.04811867]]


# Comparing to the mean_pooling model

In [59]:
mean_embedding = existing_model.encode(sentences)
print(mean_embedding)

[[ 0.05568264  0.07488462 -0.0569736  ... -0.0319594   0.02177089
  -0.01300419]
 [ 0.02329546  0.04503326 -0.04492065 ... -0.02495108  0.04223553
  -0.0437062 ]
 [ 0.02012968 -0.01120749 -0.06453869 ... -0.03425869 -0.03193577
   0.04811867]]
